# Fifa Prediction model

Each training example will be a (22,9) matrix. Each row is a player, the first 11 rows are players from the home team while the last 11 rows are players from the away team. For each player, the following details are used:

1. age
2. is forward
3. is mid fielder
4. is defense
5. is goalkeeper
6. number tournaments
7. appearances
8. goals
9. average time per team

This matrix will be passed to the first layer of the neural network that should predict the 'quality' of each player in the team. There will be a ReLU activation for this network. This will be done by doing:

$$
p = xw^T + b \\
ap = max(0, p)
$$

The output of this will be a (22, 1) matrix. This will then be passed to the team wins prediction portion of the network. This has 1 layer with 121 units with softmax activation as shown below:

$$
L = 1 \\
n^{[1]} = 2 \\
z = apw + b \\
y = softmax(z)
$$

The goal is to get a model that surpasses the following benchmarks:
- 75.42% accuracy on predicting a win
- 18% accuracy for predicting exact scores

In [76]:
# Imports
import numpy as np
import numpy.typing as npt
from sklearn.model_selection import train_test_split
import math

In [77]:
def f_x(
    players: npt.NDArray, 
    w_p: npt.NDArray, 
    b_p: float, 
    w_t: npt.NDArray, 
    b_t: float
):
    """ 
    Function that will do a forward pass for a single training example in the example above.
    
    Args:
        players (ndarray) - a (22,9) numpy array with 9 features for 22 players from the home and away team
        w_p (ndarray) - a (1,9) numpy array with weights for player predictions
        b_p (scalar) - bias for player quality prediction
        w_t (ndarray) - a (121, 22) numpy array with the weights for the team score prediction
        b_t (scalar) - bias for the team score prediction
        
    Returns:
        scores (ndarray) - a (121,1) numpy array with probability of each of the score distributions
        z_t (ndarray) - cached z_t value
        a_p (ndarray) - cached a_t value,
        z_p (ndarray) - cached z_p value
    """
    z_p = np.matmul(players, w_p.T) + b_p
    a_p = np.maximum(0, z_p)
    z_t = np.matmul(w_t, a_p) + b_t
    shifted_logits = z_t - np.max(z_t, axis=0, keepdims=True)
    e_zi = np.exp(shifted_logits)
    a_t = e_zi / np.sum(e_zi, axis=0, keepdims=True)
    return (a_t, z_t, a_p, z_p)
    

In [78]:
# Forward pass is working
players_1 = np.ones((1, 9))
players_1 = np.tile(players_1, (22, 1))
w_p = np.ones((1, 9))
b_p = 0
w_t = np.ones((121, 22))
b_t = 0
result_1 = f_x(
    players_1,
    w_p,
    b_p,
    w_t,
    b_t
)
print(result_1[0])

[[0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00

In [79]:
def L(y_pred, y):
    """
    Returns the means squared error of a single example
    
    Args:
        y_pred (ndarray) - predicted home away goal scores
        y (ndarray) - actual home away goal scores
        
    Returns:
        error (scalar) - mean squared error loss
    """
    loss = np.where(y == 1, -np.log(y_pred + 1e-15), 0) # Adding by a small number to prevent trying to get log of 0
    return np.sum(loss)

In [80]:
print(L(np.array([0.778,0.232]), np.array([1,0])))

0.25102875480374415


In [81]:
def calculate_gradients(
    X: npt.NDArray, 
    w_p: npt.NDArray, 
    b_p: float, 
    w_t: npt.NDArray, 
    b_t: float,
    Y: npt.NDArray
):
    """ 
    Function to calculate the gradients used in gradient descent
    
    Args:
        X (ndarray) - a (m,22,9) array with m training examples
        Y (ndarray) - a (m, 121, 1) array with labels for m training examples
        w_p (ndarray) - weights for the player section of neural network
        b_p (scalar) - bias for the player section of neural network
        w_t (ndarray) - weights for team section of neural network
        b_t (scalar) - bias for team section of neural network
        
    Returns:
        gradients (list) - list of gradients of parameters
    """
    dw_t = np.zeros(w_t.shape)
    db_t = 0
    dw_p = np.zeros(w_p.shape)
    db_p = 0
    J = 0
    
    m = X.shape[0]
    
    # For each training example
    for i in range(m):
        # Get the prediction
        x_i = X[i]
        y_i = Y[i]
        (y_pred_i, z_t_i, a_p_i, z_p_i) = f_x(
            x_i,
            w_p,
            b_p,
            w_t,
            b_t
        )
        
        
        # Get derivative
        # da_t_i = 2 * (y_pred_i - y_i)
        dz_t_i = y_pred_i - y_i
        dw_t += a_p_i.T * dz_t_i
        db_t += np.sum(dz_t_i)
        da_p_i = np.matmul(w_t.T, dz_t_i)
        dz_p_i = np.where(z_p_i < 0, 0, da_p_i)
        dw_p += np.matmul(dz_p_i.T, x_i)
        db_p += np.sum(dz_p_i)
        
        # Add loss 
        J += L(y_pred_i, y_i)
        
        
    dw_t /= m
    db_t /= m
    dw_p /= m
    db_p /= m
    J /= m
    return (dw_p, db_p, dw_t, db_t, J)

In [82]:
x_train_1 = np.ones((1, 9))
x_train_1 = np.tile(x_train_1, (22, 1))
w_p = np.ones((1, 9))
b_p = 0
w_t = np.ones((121, 22))
b_t = 0
X_train = np.array([
    x_train_1
])
Y_train = np.zeros((121, 1))
Y_train[0][0] = 1

print(calculate_gradients(
    X_train,
    w_p,
    b_p,
    w_t,
    b_t,
    Y_train
))

(array([[-2640., -2640., -2640., -2640., -2640., -2640., -2640., -2640.,
        -2640.]]), np.float64(-2639.9999999999986), array([[-8.92561983, -8.92561983, -8.92561983, ..., -8.92561983,
        -8.92561983, -8.92561983],
       [-8.92561983, -8.92561983, -8.92561983, ..., -8.92561983,
        -8.92561983, -8.92561983],
       [-8.92561983, -8.92561983, -8.92561983, ..., -8.92561983,
        -8.92561983, -8.92561983],
       ...,
       [-8.92561983, -8.92561983, -8.92561983, ..., -8.92561983,
        -8.92561983, -8.92561983],
       [-8.92561983, -8.92561983, -8.92561983, ..., -8.92561983,
        -8.92561983, -8.92561983],
       [-8.92561983, -8.92561983, -8.92561983, ..., -8.92561983,
        -8.92561983, -8.92561983]], shape=(121, 22)), np.float64(-120.00000000000001), np.float64(580.2906560171909))


In [83]:
# Splitting data into train / cross-validation / test sets
X = np.load("data/players_batch_6_2026-05-25 21:44:14.801126.npy")
Y = np.load("data/results_batch_6_2026-05-25 21:44:14.801174.npy")
X_train, X_mid, Y_train, Y_mid = train_test_split(X, Y, test_size=0.2, random_state=42)
X_cv, X_test, Y_cv, Y_test = train_test_split(X_mid, Y_mid, test_size=0.5, random_state=42)
print(X_train.shape, X_cv.shape, X_test.shape)
print(Y_train.shape, Y_cv.shape, Y_test.shape)

(560, 22, 9) (70, 22, 9) (70, 22, 9)
(560, 121, 1) (70, 121, 1) (70, 121, 1)


In [84]:
def gradient_descent(
    X: npt.NDArray, 
    w_p: npt.NDArray, 
    b_p: float, 
    w_t: npt.NDArray, 
    b_t: float,
    Y: npt.NDArray,
    alpha: float,
    num_iters: int
):
    """ 
    This function will minimize the cost of the model using gradient descent and return found weights
    
    Args:
        X (ndarray) - a (m, 22, 9) array serving as input to model
        w_p (ndarray) - a (1,9) array with the initial weights of the player section of model
        b_p (scalar) - initial bias for player section of model
        w_t (ndarray) - a (2,22) array with initial weights of the team section of model
        b_t (scalar) - initial bias for team section of model
        Y (ndarray) - a (m, 2, 1) array with target outputs for model
        alpha (scalar) - learning rate
        num_iters (scalar) - number of times to run gradient descent
        
    Return:
        w_p (ndarray) - a (1, 9) array that minimizes cost
        b_p (scalar) - bias that minimized cost
        w_t (ndarray) - a (2, 22) array that minimizes cost
        b_t (scalar) - bias that minimizes cost
    """
    for i in range(num_iters):
        # Get derivatives
        (dw_p, db_p, dw_t, db_t, J) = calculate_gradients(
            X=X,
            w_p=w_p,
            b_p=b_p,
            w_t=w_t,
            b_t=b_t,
            Y=Y
        )
        
        # Update weights
        w_p = w_p - (alpha * dw_p)
        w_t = w_t - (alpha * dw_t)
        b_p = b_p - (alpha * db_p)
        b_t = b_t - (alpha * db_t)
        
        # Print result
        print(f"Cost at epoch {i}/{num_iters} = {J}")
        
    return (w_p, w_t, b_p, b_t)

In [85]:
# Get initial weights
w_t = np.random.rand(121, 22)
w_p = np.random.rand(1, 9)
b_p = 0
b_t = 0

# Run gradient descent
(new_w_p, new_w_t, new_b_p, new_bt) = gradient_descent(
    X=X_train,
    w_p=w_p,
    w_t=w_t,
    b_p=b_p,
    b_t=b_t,
    Y=Y_train,
    alpha=0.01,
    num_iters=100
)

Cost at epoch 0/100 = 34.538776286548035
Cost at epoch 1/100 = 4.8236133067090865
Cost at epoch 2/100 = 4.820678201435731
Cost at epoch 3/100 = 4.817851800467333
Cost at epoch 4/100 = 4.815127178126957
Cost at epoch 5/100 = 4.812497732397559
Cost at epoch 6/100 = 4.809957300599308
Cost at epoch 7/100 = 4.807500279383904
Cost at epoch 8/100 = 4.805121818153152
Cost at epoch 9/100 = 4.80281815318223
Cost at epoch 10/100 = 4.800587202735956
Cost at epoch 11/100 = 4.798429664467577
Cost at epoch 12/100 = 4.796351108297274
Cost at epoch 13/100 = 4.794366037839173
Cost at epoch 14/100 = 4.792505560466153
Cost at epoch 15/100 = 4.790829936383503
Cost at epoch 16/100 = 4.7894398597783585
Cost at epoch 17/100 = 4.788454553586348
Cost at epoch 18/100 = 4.787914493217809
Cost at epoch 19/100 = 4.787700333075138
Cost at epoch 20/100 = 4.787635370235726
Cost at epoch 21/100 = 4.787615990302318
Cost at epoch 22/100 = 4.787607282050122
Cost at epoch 23/100 = 4.787600739639531
Cost at epoch 24/100 = 4

In [86]:
def J(
    X: npt.NDArray, 
    w_p: npt.NDArray, 
    b_p: float, 
    w_t: npt.NDArray, 
    b_t: float,
    Y: npt.NDArray
):
    """ 
    Evaluate model on dataset
    
    Args:
        X (ndarray) - a (m,22,9) array with m examples
        Y (ndarray) - a (m, 2, 1) array with labels for m examples
        w_p (ndarray) - weights for the player section of neural network
        b_p (scalar) - bias for the player section of neural network
        w_t (ndarray) - weights for team section of neural network
        b_t (scalar) - bias for team section of neural network
    """
    J = 0
    m = X.shape[0]
    
    # For each training example
    for i in range(m):
        # Get the prediction
        x_i = X[i]
        y_i = Y[i]
        (y_pred_i, _, _, _) = f_x(
            x_i,
            w_p,
            b_p,
            w_t,
            b_t
        )
        
        # Add loss 
        J += L(y_pred_i, y_i)
        
    J /= m
    print(f"Cost is {J}")
    

In [87]:
J(
    X=X_cv,
    w_p=new_w_p,
    b_p=new_b_p,
    w_t=new_w_t,
    b_t=new_bt,
    Y=Y_cv
)

Cost is 4.8537600833362005


In [143]:
def interpret_probabilities(prob):
    """ 
    Function to interpret the returned probabilities by the model
    
    Args:
        prob (ndarray) - array with probabilities for different match results
        
    Returns:
        scores (str) - human readable version of prediction
    """
    predicted = np.argmax(prob)
    home_score = math.floor(predicted / 11)
    away_score = predicted % 11
    return f"{home_score}-{away_score}"

In [144]:
prob = f_x(
    X_cv[5],
    new_w_p,
    new_b_p,
    new_w_t,
    new_bt
)[0]

predicted_score = interpret_probabilities(prob)
actual_score = interpret_probabilities(Y_cv[5])
print(f"Predicted score is {predicted_score} while actual is {actual_score}")

Predicted score is 0-0 while actual is 5-0


In [145]:
def accuracy_score(y_pred, y_label):
    """ 
    Return the ratio of correct responses across the entire test set.add
    
    Args:
        y_pred (ndarray): (m, 121, 1) array with predictions for each class from the model from the model
        y_label(ndarray): (m, 121, 1) array with the actual labels
        
    Returns:
        accuracy (scalar): ratio of correct responses to all responses
    """
    predictions = np.argmax(y_pred, axis=1)
    actual = np.argmax(y_label, axis=1)
    num_correct = np.sum(predictions == actual)
    return num_correct / y_pred.shape[0]

In [146]:
def predict(
    X: npt.NDArray, 
    w_p: npt.NDArray, 
    b_p: float, 
    w_t: npt.NDArray, 
    b_t: float,
):
    """ 
    Make prediction for all of the examples in the passed dataset
    
    Args:
        X (ndarray) - a (m,22,9) array with m examples
        w_p (ndarray) - weights for the player section of neural network
        b_p (scalar) - bias for the player section of neural network
        w_t (ndarray) - weights for team section of neural network
        b_t (scalar) - bias for team section of neural network
        
    Returns:
        Y_pred (ndarray) - a (m, 121, 1) array with prediction of all examples in X
    """
    m = X.shape[0]
    predictions = []
    
    for i in range(m):
        x_i = X[i]
        (pred, _, _, _) = f_x(
            x_i,
            w_p,
            b_p,
            w_t,
            b_t
        )
        predictions.append(pred)
        
    Y_pred = np.array(predictions)
    return Y_pred

In [147]:
y_pred = predict(
    X_cv,
    new_w_p,
    new_b_p,
    new_w_t,
    new_bt
)
accuracy = accuracy_score(y_pred, Y_cv)
print(f"Accuracy of model is {accuracy}")

Accuracy of model is 0.07142857142857142


In [154]:
class PredictionRecall:
    def __init__(self, t_p: int, f_p: int, f_n:int, total: int, label: str):
        """ 
        Return object with prediction recall values
        
        Args:
            label (str): Name of the class
            t_p (int): Total true positive for the given label
            f_p (int): Total false positives for given label
            total (int): Total items with given label
            f_n (int): Total false negatives for given label
        """
        self.total = total
        self.t_p = t_p
        self.f_p = f_p
        self.f_n = f_n
        self.label = label
        
    def __repr__(self):
        return f"{self.label} with t_p: {self.t_p}, f_p: {self.f_p}, f_n: {self.f_n} and total: {self.total}"

def get_precision_recall(y_pred, y_label) -> list[PredictionRecall]:
    """ 
    Returns the precision and recall of the model based on the predictions it has made
    
    Args:
        y_pred (ndarray) - a (m, 121, 1) array with the predictions of the model
        y_label (ndarray) - a (m, 121, 1) array with the actual values
        
    Returns:
        prediction_recall (list) - list of prediction recall values
    """
    predictions = [interpret_probabilities(prob) for prob in y_pred]
    actual = [interpret_probabilities(prob) for prob in y_label]
    all_items = predictions.copy()
    all_items.extend(actual)
    labels = set(all_items)
    
    m = len(actual)
    precisions = []
    
    for label in labels:
        total = 0
        false_positive = 0
        false_negative = 0
        true_positive = 0
        
        for i in range(m):
            entry = actual[i]
            pred = predictions[i]
            
            if entry == label:
                total += 1
                
            if entry == label and pred == entry:
                true_positive += 1
            elif entry != label and pred == label:
                false_positive += 1
        false_negative = total - true_positive
        
        precisions.append(PredictionRecall(
            t_p=true_positive,
            f_p=false_positive,
            f_n=false_negative,
            total=total,
            label=label
        ))
    
    return precisions
    

In [155]:
precisions = get_precision_recall(
    y_pred=np.array([
        [
            [0],
            [0],
            [0]
        ],
        [
            [0],
            [0],
            [0]
        ],
        [
            [0],
            [0],
            [0]
        ],
        [
            [0],
            [0],
            [0]
        ],
        [
            [0],
            [0],
            [0]
        ],
        [
            [0],
            [0],
            [0]
        ],
        [
            [0],
            [0],
            [0]
        ],
        [
            [0],
            [0],
            [0]
        ],
        [
            [0],
            [0],
            [0]
        ]]),
    y_label=np.array([
        [
            [0],
            [0],
            [0]
        ],
        [
            [0],
            [0],
            [0]
        ],
        [
            [0],
            [1],
            [0]
        ],
        [
            [0],
            [0],
            [2]
        ],
        [
            [0],
            [1],
            [0]
        ],
        [
            [0],
            [1],
            [0]
        ],
        [
            [0],
            [0],
            [2]
        ],
        [
            [0],
            [0],
            [0]
        ],
        [
            [0],
            [1],
            [0]
        ]]),
)
print(precisions)

[0-2 with t_p: 0, f_p: 0, f_n: 2 and total: 2, 0-1 with t_p: 0, f_p: 0, f_n: 4 and total: 4, 0-0 with t_p: 3, f_p: 6, f_n: 0 and total: 3]


In [164]:
def precision_recall(y_pred, y_label):
    """ 
        Print precision recall for model predictions
        
        Args:
            y_pred (ndarray): a (m, 121, 1) array with the predictions of the model
            y_label (ndarray): a (m, 121, 1) array with labels of the model
    """ 
    precisions = get_precision_recall(y_pred=y_pred, y_label=y_label)
    print(f"    Precision and recall")
    for prec in precisions:
        precision = 0 if prec.f_p + prec.t_p == 0 else (prec.t_p / (prec.f_p + prec.t_p))
        recall = 0 if prec.t_p + prec.f_n == 0 else (prec.t_p) / (prec.t_p + prec.f_n)
        p_1 = 0 if precision == 0 else 1 / precision
        r_1 = 0 if recall == 0 else 1 / recall
        f1_score = 1 / 0.5 * (p_1 + r_1)
        print(f"{prec.label.center(5)}  TP:{prec.t_p}  FP:{prec.f_p}  FN:{prec.f_n}: Prec:{precision:.4f}  Rec:{recall:.4f}  F1:{f1_score:.4f}  Total:{prec.total}")

In [165]:
precision_recall(y_pred=y_pred, y_label=Y_cv)

    Precision and recall
 2-0   TP:0  FP:1  FN:20: Prec:0.0000  Rec:0.0000  F1:0.0000  Total:20
 0-0   TP:5  FP:64  FN:0: Prec:0.0725  Rec:1.0000  F1:29.6000  Total:5
 5-0   TP:0  FP:0  FN:6: Prec:0.0000  Rec:0.0000  F1:0.0000  Total:6
 1-0   TP:0  FP:0  FN:14: Prec:0.0000  Rec:0.0000  F1:0.0000  Total:14
 6-0   TP:0  FP:0  FN:1: Prec:0.0000  Rec:0.0000  F1:0.0000  Total:1
 3-0   TP:0  FP:0  FN:11: Prec:0.0000  Rec:0.0000  F1:0.0000  Total:11
 4-0   TP:0  FP:0  FN:13: Prec:0.0000  Rec:0.0000  F1:0.0000  Total:13
